<a href="https://colab.research.google.com/github/varshacode01/varshacode01/blob/main/SelfWrap_Material_Failure_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Predicting Early Material Failure in SelfWrap (Quality Engineering)**

 This model predicts early material failure using simulated degradation test data. The model shows how porosity, thickness, and temperature influence mechanical integrity. This could help Venostent flag high-risk batches early and optimize material parameters

 Goal:
Use dummy degradation test data to predict which material samples will fail early (lose strength too fast)

In [ ]:
# Predicting Early Material Failure
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

# Simulate dummy data
np.random.seed(42)
n = 200
data = pd.DataFrame({
    "thickness_mm": np.random.uniform(0.2, 0.5, n),
    "porosity_pct": np.random.uniform(30, 60, n),
    "test_temp_c": np.random.choice([35, 37, 39], n),
    "degradation_weeks": np.random.uniform(4, 12, n)
})

# Simulate strength drop
data["tensile_strength_mpa"] = (
    80 * data["thickness_mm"] -
    0.5 * data["porosity_pct"] -
    2 * (data["test_temp_c"] - 37) -
    3 * (12 - data["degradation_weeks"]) +
    np.random.normal(0, 5, n)
)

# Define failure: < 20 MPa strength = early failure
data["early_failure"] = (data["tensile_strength_mpa"] < 20).astype(int)

# Split + train model
X = data[["thickness_mm", "porosity_pct", "test_temp_c", "degradation_weeks"]]
y = data["early_failure"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)

# Results
print(classification_report(y_test, preds))
sns.heatmap(confusion_matrix(y_test, preds), annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Early Failure Prediction")
plt.show()

# Feature importance
feat_imp = pd.Series(model.feature_importances_, index=X.columns)
feat_imp.sort_values().plot(kind='barh', color='teal')
plt.title("Feature Importance - Early Failure Risk")
plt.show()
